# Job Market Intelligence

## Notebook 01 — Exploratory Data Analysis & NLP Feature Engineering

### Objectives

- Load and inspect the cleaned dataset.
- Assess data quality.
- Explore job market characteristics.
- Prepare job descriptions for NLP.
- Extract structured technical skills.
- Export the processed dataset for dashboarding and machine learning.

In [6]:
from pathlib import Path
import sys

PROJECT_ROOT = Path().resolve().parent
sys.path.append(str(PROJECT_ROOT))

In [7]:
import pandas as pd
from dotenv import load_dotenv
from src.utils.logger import get_logger
from src.data.db import get_db_connection

In [8]:
load_dotenv()
logger = get_logger(__name__)

conn = None

try:
    logger.info("Start connecting to the database")
    conn = get_db_connection()
    logger.info("Connected to the database successfully")

    select_query = "SELECT * FROM clean_job_posting_v2"

    logger.info("Fetching the query into a DataFrame")
    df = pd.read_sql(select_query, conn)
    logger.info("Data loaded successfully")

except Exception as e:
        logger.error(f" failed: {e}")

finally:
    if conn is not None:
        conn.close()
        logger.info("Database connection closed")        

2026-07-05 21:55:41 | INFO | __main__ | Start connecting to the database
2026-07-05 21:55:41 | INFO | src.data.db | Connecting to PostgreSQL


2026-07-05 21:55:48 | INFO | src.data.db | PostgreSQL connection established
2026-07-05 21:55:48 | INFO | __main__ | Connected to the database successfully
2026-07-05 21:55:48 | INFO | __main__ | Fetching the query into a DataFrame
C:\Users\anton\AppData\Local\Temp\ipykernel_20268\3353679563.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(select_query, conn)
2026-07-05 21:56:02 | INFO | __main__ | Data loaded successfully
2026-07-05 21:56:02 | INFO | __main__ | Database connection closed


# 1. Initial Data Loading and Structural Inspection

## 1.1 Dataset Preview

In [9]:
df.head()

,standardized_job_title,clean_job_title,company_name,job_country,job_location,employment_type,is_remote,job_description,job_source,posted_at_utc
0,data analyst,(usa) data analyst ii,Vizio,US,NaN,full-time,False,"Position Summary...As a Data Analyst II, you w...",ziprecruiter,2026-06-10 00:00:00+00:00
1,data analyst,"(usa) senior, data analyst",Walmart,US,NaN,full-time,False,Position Summary...\n\nWhat you'll do...Senior...,ziprecruiter,2026-06-11 00:00:00+00:00
2,data scientist,<new graduate hiring> data scientist,NXP Semiconductors,SG,NaN,full-time,False,Data Scientist\n\nAbout the Role\n\nWe are see...,built in,2026-06-01 00:00:00+00:00
3,business intelligence analyst,advanced business intelligence analyst waterdown,L3Harris Technologies,CA,NaN,full-time,False,Location: Northwest Newfoundland/Eastern Labra...,learn4good,2026-06-09 00:00:00+00:00
4,data analyst,afc data analyst,AtkinsRéalis,SA,NaN,full-time,False,Overview\n\nWe are seeking an experienced AFC ...,atkinsréalis careers,2026-06-09 00:00:00+00:00


## 1.2 Dataset Shape

In [10]:
df.shape

(1044, 10)

## 1.3 Column Names

In [11]:
df.columns.tolist()

['standardized_job_title',
 'clean_job_title',
 'company_name',
 'job_country',
 'job_location',
 'employment_type',
 'is_remote',
 'job_description',
 'job_source',
 'posted_at_utc']

## 1.4 Data Types

In [12]:
df.dtypes

standardized_job_title                    str
clean_job_title                           str
company_name                              str
job_country                               str
job_location                              str
employment_type                           str
is_remote                                bool
job_description                           str
job_source                                str
posted_at_utc             datetime64[us, UTC]
dtype: object

## 1.5 DataFrame Info

In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1044 entries, 0 to 1043
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype              
---  ------                  --------------  -----              
 0   standardized_job_title  1044 non-null   str                
 1   clean_job_title         1044 non-null   str                
 2   company_name            1044 non-null   str                
 3   job_country             982 non-null    str                
 4   job_location            63 non-null     str                
 5   employment_type         1037 non-null   str                
 6   is_remote               1044 non-null   bool               
 7   job_description         1044 non-null   str                
 8   job_source              1044 non-null   str                
 9   posted_at_utc           736 non-null    datetime64[us, UTC]
dtypes: bool(1), datetime64[us, UTC](1), str(8)
memory usage: 74.6 KB


In [14]:
df.isnull().sum().sort_values(ascending=False)

job_location              981
posted_at_utc             308
job_country                62
employment_type             7
clean_job_title             0
standardized_job_title      0
company_name                0
is_remote                   0
job_description             0
job_source                  0
dtype: int64

In [15]:
df[df['is_remote']]['job_country'].isnull().sum()

np.int64(49)

## 1.6 Dataset Describe

In [16]:
df.describe(include='all').T

,count,unique,top,freq,mean,min,25%,50%,75%,max
standardized_job_title,1044,9,ai engineer,224,NaN,NaN,NaN,NaN,NaN,NaN
clean_job_title,1044,862,data scientist,19,NaN,NaN,NaN,NaN,NaN,NaN
company_name,1044,800,Micron Technology,13,NaN,NaN,NaN,NaN,NaN,NaN
job_country,982,11,SG,150,NaN,NaN,NaN,NaN,NaN,NaN
job_location,63,14,Anywhere,49,NaN,NaN,NaN,NaN,NaN,NaN
employment_type,1037,8,full-time,954,NaN,NaN,NaN,NaN,NaN,NaN
is_remote,1044,2,False,995,NaN,NaN,NaN,NaN,NaN,NaN
job_description,1044,1034,Company Overview:\n\nUK residents looking for ...,4,NaN,NaN,NaN,NaN,NaN,NaN
job_source,1044,297,bebee,101,NaN,NaN,NaN,NaN,NaN,NaN
posted_at_utc,736,NaN,NaN,NaN,2026-06-07 14:40:21.195652+00:00,2026-05-16 00:00:00+00:00,2026-06-04 00:00:00+00:00,2026-06-10 00:00:00+00:00,2026-06-13 00:00:00+00:00,2026-06-16 00:00:00+00:00


# 2. Data Quality Assessment

## 2.1 Missing Values Analysis

### Observations

- Most columns contain complete data, indicating good overall dataset quality.
- `job_description` has no missing values, which is important for the NLP and skill extraction phases.
- `posted_at_utc` contains the highest number of missing values (~37%), which may limit future time-series and trend analysis.
- `job_country` contains a small number of missing values. A recovery process was implemented using `job_location` and an ISO Alpha-2 country reference table, reducing the number of missing values where reliable country information was available. The remaining missing values correspond to remote jobs (`job_location = "Anywhere"`) or locations that do not explicitly specify a country. These values were intentionally left as `NULL` to preserve data quality rather than introducing unsupported assumptions.
- `employment_type` contains only a few missing values and will be investigated separately to determine whether they can be reliably recovered from the available job metadata.

In [17]:
null_values = df.isnull().sum()
percentage_null = (df.isnull().sum() / df.shape[0]) * 100

missing_values_summary = pd.DataFrame({
    'missing_count': null_values,
    'missing_percentage': percentage_null
})

missing_values_summary


,missing_count,missing_percentage
standardized_job_title,0,0.000000
clean_job_title,0,0.000000
company_name,0,0.000000
job_country,62,5.938697
job_location,981,93.965517
employment_type,7,0.670498
is_remote,0,0.000000
job_description,0,0.000000
job_source,0,0.000000
posted_at_utc,308,29.501916


## 2.2 Duplicate Records Analysis

### Observations

- No duplicate rows were detected in the dataset.
- This indicates that the data cleaning or ingestion process successfully avoided exact duplicate records.
- As a result, there is no immediate need for duplicate removal at this stage.

In [18]:
print(df.duplicated().sum())

0


## 2.3 Data Type Optimization

### Observations

- Converted low-cardinality columns to categorical type.
- This improves memory efficiency and performance for grouping operations.
- Text-heavy and high-cardinality columns were kept as object type.

In [19]:
cols = ['standardized_job_title', 'job_country', 'employment_type', 'job_source']
df[cols] = df[cols].astype('category')

# 3. Univariate Analysis

## 3.1 Job Role Distribution

### Observations

- The dataset contains a diverse range of AI and data-related job roles, providing broader coverage of the current job market.
- AI Engineer is the most frequently occurring role, followed by Data Scientist, Data Engineer, Data Analyst, Machine Learning Engineer, and Business Intelligence Analyst.
- A few specialized roles, such as Analytics Engineer, Data Architect, and MLOps Engineer, have relatively few observations, indicating lower representation in the current dataset.
- The distribution reflects the search strategy and current job market demand rather than a balanced sample across all job roles.
- The dataset provides sufficient coverage of the major AI and data professions for exploratory analysis, dashboard development, and machine learning tasks, while additional data collection would further improve the representation of less common roles.

In [20]:
df['standardized_job_title'].value_counts()

standardized_job_title
ai engineer                      224
data scientist                   189
data engineer                    183
data analyst                     150
machine learning engineer        144
business intelligence analyst    138
analytics engineer                 8
data architect                     6
mlops engineer                     2
Name: count, dtype: int64

## 3.2 Job Country Distribution

### Observations

- The dataset includes job postings from multiple countries across North America, Europe, Asia, and the Middle East, providing a broader view of the AI and data job market.
- Singapore, India, the United Kingdom, the United States, and Canada represent the largest share of job postings in the dataset.
- Middle Eastern countries, including Saudi Arabia, Qatar, Bahrain, Oman, and Kuwait, are also well represented, supporting regional market analysis.
- Ireland currently contains only a single job posting, indicating limited representation in the current dataset.
- The country distribution reflects the selected search strategy and data availability rather than an equal sampling across all countries.

In [21]:
df['job_country'].value_counts()

job_country
SG    150
IN    141
GB    134
US    130
CA    116
SA    110
QA    101
BH     43
OM     30
KW     26
IE      1
Name: count, dtype: int64

## 3.3 Employment Type Distribution

### Observations

- Full-time positions dominate the dataset, accounting for the vast majority of job postings.
- Contractor, internship, and part-time opportunities are available but represent a much smaller portion of the dataset.
- A small number of postings advertise multiple employment types (e.g., full-time and contractor), reflecting the flexibility offered by some employers.
- The employment type distribution indicates that the current AI and data job market is primarily focused on full-time hiring.
- The distribution reflects the available job postings collected from the selected data sources and should not be interpreted as a balanced representation of all employment types.

In [22]:
df['employment_type'].value_counts()

employment_type
full-time                               954
contractor                               39
internship                               16
part-time                                14
full-time and contractor                  6
full-time and part-time                   6
full-time and internship                  1
full-time, part-time, and contractor      1
Name: count, dtype: int64

## 3.4 Remote Work Distribution

### Observations

- The majority of job postings are classified as on-site, while a smaller proportion are fully remote.
- Approximately 4.7% of the collected job postings are remote positions, indicating that remote opportunities are available but less common in the current dataset.
- The distribution reflects the information provided by the job posting sources and the selected search strategy.
- Remote jobs were preserved in the dataset to enable future analysis of remote work trends across different AI and data-related roles.
- Since remote positions often do not specify a physical country (`job_location = "Anywhere"`), some records intentionally retain missing `job_country` values to preserve data integrity.

In [23]:
df['is_remote'].value_counts()

is_remote
False    995
True      49
Name: count, dtype: int64

## 3.5 Job Description Quality Check

### Observations

- Job descriptions are present for all records (no missing values)
- Job descriptions are generally detailed and contain valuable information such as responsibilities, required skills, technologies, and qualifications.
- The text appears sufficiently rich and structured, making it suitable for NLP tasks such as skill extraction
- No obvious noise or extremely short descriptions were observed

In [24]:
pd.set_option('display.max_colwidth',None)

print(f"Missing values:", df['job_description'].isnull().sum())
print(f"Total rows :", len(df['job_description']))

df['job_description'].head(5)

Missing values: 0
Total rows : 1044


0                                                                                                                                                                                                                                                                                                                                                                                                                Position Summary...As a Data Analyst II, you will leverage advanced analytical techniques and business knowledge to support data-driven decision-making across multiple domains. This role involves identifying relevant data sources, ensuring data quality, and applying statistical methods to generate actionable insights for VIZIO's Inscape clients. You will collaborate with stakeholders to develop visualizations and reports that effectively communicate findings, while supporting process improvements and project delivery. The position requires proficiency in data management, exploratory analysis,

# 4. Bivariate Analysis

## 4.1 Job Role Distribution by Country

### Observations

- AI Engineer, Data Scientist, Data Engineer, Data Analyst, Machine Learning Engineer, and Business Intelligence Analyst are consistently represented across most countries.
- The distribution of job roles varies by country, with no single role dominating across all regions.
- Singapore, the United States, India, the United Kingdom, Canada, Saudi Arabia, and Qatar exhibit the broadest representation of AI and data-related roles.
- Specialized roles such as Analytics Engineer, Data Architect, and MLOps Engineer appear infrequently across all countries, reflecting their lower representation in the current dataset.
- Overall, the dataset captures a diverse distribution of AI and data roles across multiple countries, enabling meaningful comparative analysis between regional job markets.

In [25]:
job_country_role = pd.crosstab(
    df['job_country'],
    df['standardized_job_title']
)

job_country_role

standardized_job_title,ai engineer,analytics engineer,business intelligence analyst,data analyst,data architect,data engineer,data scientist,machine learning engineer,mlops engineer
job_country,,,,,,,,,
BH,15,1,4,6,0,7,6,4,0
CA,23,0,23,1,0,20,27,22,0
GB,27,1,12,21,0,25,24,23,1
IE,0,0,0,0,0,0,0,1,0
IN,30,1,22,18,1,26,21,22,0
KW,2,1,5,11,0,1,6,0,0
OM,11,0,5,4,0,3,5,2,0
QA,25,1,12,21,1,20,14,7,0
SA,26,0,12,18,2,20,21,11,0


## 4.2 Employment Type by Job Role

### Observations

- Full-time employment is the dominant hiring model across all AI and data-related job roles.
- Contractor, internship, and part-time opportunities are available but occur much less frequently than full-time positions.
- AI Engineer, Data Scientist, Data Engineer, Data Analyst, Machine Learning Engineer, and Business Intelligence Analyst all exhibit a similar employment pattern, with full-time positions representing the majority of postings.
- A small number of job postings advertise multiple employment types (e.g., full-time and contractor), reflecting flexible hiring arrangements offered by some employers.
- Overall, the employment type distribution suggests that the AI and data job market is primarily driven by full-time opportunities across all major job roles.

In [36]:
job_employment_type = pd.crosstab(
    df['standardized_job_title'],
    df['employment_type']
)

job_employment_type

employment_type,contractor,full-time,full-time and contractor,full-time and internship,full-time and part-time,"full-time, part-time, and contractor",internship,part-time
standardized_job_title,,,,,,,,
ai engineer,12,201,2,1,2,0,4,1
analytics engineer,0,6,0,0,1,0,0,1
business intelligence analyst,0,133,0,0,0,0,5,0
data analyst,10,126,2,0,0,0,3,8
data architect,1,5,0,0,0,0,0,0
data engineer,6,166,2,0,1,0,2,1
data scientist,3,181,0,0,1,1,1,2
machine learning engineer,6,135,0,0,1,0,1,1
mlops engineer,1,1,0,0,0,0,0,0


## 4.3 Remote Work Distribution by Job Role

### Observations

- On-site positions represent the majority of job postings across all AI and data-related roles.
- Remote opportunities are available for most major job roles, including AI Engineer, Data Scientist, Data Engineer, Data Analyst, Machine Learning Engineer, and Business Intelligence Analyst.
- Data Engineer, Data Scientist, and Data Analyst currently have the highest number of remote job postings in the dataset.
- Analytics Engineer and MLOps Engineer do not contain remote opportunities in the current dataset, although this may be influenced by their limited number of observations.
- Overall, the dataset indicates that while remote work is available across the AI and data job market, employers continue to predominantly advertise on-site positions.

In [27]:
remote_job_role = pd.crosstab(
    df['standardized_job_title'],
    df['is_remote']
)
remote_job_role

is_remote,False,True
standardized_job_title,,
ai engineer,216,8
analytics engineer,8,0
business intelligence analyst,137,1
data analyst,140,10
data architect,6,0
data engineer,170,13
data scientist,177,12
machine learning engineer,139,5
mlops engineer,2,0


# 5. Preparing for NLP Analysis

## 5.1 Identifying Relevant Text Columns for NLP

### Selected Column for NLP

- job_description

### Justification

- The column contains no missing values, ensuring complete coverage across all records
- It includes rich and detailed textual information such as responsibilities, required skills, and qualifications
- The content is directly relevant to the business objective of extracting skills and tools from job postings
- Based on EDA, the text quality is sufficient for downstream NLP tasks

## 5.2 Potential Challenges in NLP Processing

### Potential Challenges

- Inconsistent representation of technical terms (e.g., Python, python, PYTHON), requiring text normalization before skill extraction.
- Job descriptions are unstructured, with skills embedded within paragraphs rather than listed explicitly
- Presence of noise or irrelevant text (e.g., company descriptions or general statements) that may interfere with accurate skill extraction

## 5.3 Defining the NLP Objective

### NLP Objective

- Extract key technical skills, tools, and technologies mentioned in job descriptions
- Associate the extracted skills, tools, and technologies with each standardized AI and data-related job role.
- Examples include programming languages (Python, SQL, R), tools (Power BI, Tableau), and technologies (machine learning frameworks)

### Business Purpose

- Enable identification of the most in-demand skills for each job role
- Help job seekers understand which skills and tools they need to learn for specific roles
- Assist recruiters in identifying the key competencies required for each role
- Support the development of a data-driven dashboard that summarizes job market requirements

## 5.4 Initial Skill Extraction Approach

### Skill Extraction Approach

- Use a dictionary-based method where we define a list of known skills, tools, and technologies
- The extraction process will search for these predefined skills within the job descriptions

### Why This Approach

- It is simple and easy to implement without requiring complex NLP models
- It provides full control over what is being extracted
- It is easy to debug and validate results
- It provides a transparent and reproducible baseline that can later be extended with more advanced NLP techniques as the dataset grows.

## 5.5 Initial Skill Dictionary

## Defined Skill Categories

### Programming Languages

- Python
- SQL
- R
- Scala
- Java
- C++
- Bash

### Data Visualization & BI Tools

- Power BI
- Tableau
- Excel
- Google Sheets
- Looker
- Qlik Sense

### Databases & Data Warehouses

- PostgreSQL
- MySQL
- SQL Server
- Oracle
- MongoDB
- Snowflake
- BigQuery
- Amazon Redshift

### Cloud Platforms

- Amazon Web Services (AWS)
- Microsoft Azure
- Google Cloud Platform (GCP)

### Data Engineering & DevOps Tools

- Apache Airflow
- dbt
- Docker
- Kubernetes
- Git
- GitHub
- Jenkins

### Machine Learning & AI Frameworks

- Scikit-learn
- TensorFlow
- PyTorch
- XGBoost
- LightGBM
- Keras

### Big Data Technologies

- Apache Spark
- Hadoop
- Apache Kafka
- Databricks

### AI Technologies & Concepts

- Machine Learning
- Artificial Intelligence
- Deep Learning
- Natural Language Processing (NLP)
- Large Language Models (LLMs)
- Computer Vision
- Generative AI
- Prompt Engineering
- Retrieval-Augmented Generation (RAG)
- MLOps

### Observations

- The initial dictionary has been expanded to cover a broader range of AI and data-related roles, including Data Engineering, Machine Learning, Business Intelligence, Analytics Engineering, and AI Engineering.
- Skills are organized into logical categories to simplify extraction, analysis, and future maintenance of the NLP pipeline.
- The dictionary focuses on commonly requested programming languages, tools, platforms, frameworks, and technologies frequently mentioned in AI and data job postings.
- This dictionary serves as the baseline for dictionary-based skill extraction and will be iteratively refined as new technologies and competencies are identified during analysis.

In [28]:
skill_dictionary = {
    "programming_languages": [
        "python", "sql", "r", "scala", "java", "c++", "bash"
    ],

    "bi_tools": [
        "power bi", "tableau", "excel", "google sheets",
        "looker", "qlik sense"
    ],

    "databases": [
        "postgresql", "mysql", "sql server", "oracle",
        "mongodb", "snowflake", "bigquery", "redshift"
    ],

    "cloud": [
        "aws", "azure", "gcp"
    ],

    "data_engineering": [
        "apache airflow", "dbt", "docker",
        "kubernetes", "git", "github", "jenkins"
    ],

    "machine_learning": [
        "scikit-learn", "tensorflow", "pytorch",
        "xgboost", "lightgbm", "keras"
    ],

    "big_data": [
        "apache spark", "hadoop", "kafka", "databricks"
    ],

    "ai_technologies": [
        "machine learning",
        "artificial intelligence",
        "deep learning",
        "natural language processing",
        "nlp",
        "large language models",
        "llm",
        "computer vision",
        "generative ai",
        "prompt engineering",
        "retrieval-augmented generation",
        "rag",
        "mlops"
    ]
}

# 6. NLP Skill Extraction Implementation

## 6.1 Text Normalization

### Why Text Normalization is Important

- Technical skills, tools, and technologies may appear in different formats, such as uppercase, lowercase, or mixed case (e.g., Python, python, PYTHON).
- Without normalization, identical terms may be treated as different values, reducing the accuracy of the extraction process.
- Converting text to lowercase standardizes the job descriptions and improves the consistency of dictionary-based skill matching.
- Text normalization serves as the foundation for reliable extraction of programming languages, tools, frameworks, databases, cloud platforms, and AI technologies.

## 6.1.1 Creating a Normalized Text Column

In [29]:
df['normalized_job_description'] = df['job_description'].str.lower()

## 6.1.2 Creating a Skill Extraction Function

### Observations

- The skill extraction logic was implemented as a reusable function that can extract terms from multiple predefined skill categories.
- Separate columns were automatically generated for each category (e.g., programming languages, BI tools, databases, cloud platforms, data engineering, machine learning, big data, and AI technologies), making the extracted information easier to analyze.
- The sample output confirms that the dictionary-based approach successfully identifies relevant technical terms from job descriptions.
- The categorized extraction structure improves the scalability and maintainability of the NLP pipeline, allowing new skill categories or terms to be added with minimal code changes.
- Some technologies may still be missed if they appear in alternative spellings or abbreviations, so the dictionaries will be iteratively refined as the dataset evolves.

In [30]:
import re

def extract_terms(text, terms):
    matched_terms = []

    for term in terms:
        pattern = r'\b' + re.escape(term) + r'\b'

        if re.search(pattern, text):
            matched_terms.append(term)

    return matched_terms

In [31]:
for category, terms in skill_dictionary.items():
    column_name = f"extracted_{category}"

    df[column_name] = df["normalized_job_description"].apply(
        lambda text: extract_terms(text, terms)
    )

In [32]:
extracted_columns = [
    col for col in df.columns
    if col.startswith("extracted_")
]

df[extracted_columns].head()    

,extracted_programming_languages,extracted_bi_tools,extracted_databases,extracted_cloud,extracted_data_engineering,extracted_machine_learning,extracted_big_data,extracted_ai_technologies
0,"[python, sql, r, scala]","[power bi, tableau]",[],[],[],[],[],[]
1,"[python, sql, r, scala]","[power bi, tableau]",[],[],[],[],[],[]
2,"[python, sql, r]","[power bi, tableau]",[],"[aws, azure]",[],[scikit-learn],[],"[machine learning, large language models, generative ai]"
3,[sql],[power bi],[],[],[],[],[],[]
4,"[python, sql, r]","[power bi, tableau]",[],[],[],[],[hadoop],[machine learning]


## 6.3 Extraction Coverage Analysis

### Observations

- Programming languages achieved the highest extraction coverage (75.96%), indicating that technical programming skills are explicitly mentioned in most AI and data-related job postings.
- AI technologies also showed strong coverage (56.42%), reflecting the increasing demand for AI-related expertise across the job market.
- Categories such as cloud (37.93%), BI tools (30.17%), data engineering (23.66%), big data (20.11%), machine learning (18.77%), and databases (18.39%) have lower coverage, suggesting that these skills are either more specialized or are not consistently stated in job descriptions.
- Overall, the extraction coverage confirms that the NLP pipeline successfully generated structured skill features across all predefined categories, providing a reliable foundation for downstream Power BI analysis and machine learning tasks.

In [ ]:
# Create extraction coverage summary
coverage_summary = []

for col in extracted_columns:
    non_empty_count = df[col].apply(lambda x: len(x) > 0).sum()
    empty_count = df[col].apply(lambda x: len(x) == 0).sum()
    coverage_percentage = round((non_empty_count / len(df)) * 100, 2)

    coverage_summary.append({
        "category": col.replace("extracted_", ""),
        "non_empty_count": non_empty_count,
        "empty_count": empty_count,
        "coverage_percentage": coverage_percentage
    })
 
coverage_summary_df = pd.DataFrame(coverage_summary)

coverage_summary_df

,category,non_empty_count,empty_count,coverage_percentage
0,programming_languages,793,251,75.96
1,bi_tools,315,729,30.17
2,databases,192,852,18.39
3,cloud,396,648,37.93
4,data_engineering,247,797,23.66
5,machine_learning,196,848,18.77
6,big_data,210,834,20.11
7,ai_technologies,589,455,56.42


In [ ]:
df.to_csv("processed_job_data.csv", index=False)

# Conclusion

This notebook successfully prepared the AI and Data job market dataset for downstream analysis.

Key accomplishments include:

- Data quality assessment and validation.
- Exploratory analysis of job market characteristics.
- Construction of a structured NLP skill dictionary.
- Extraction of technical skills across eight predefined categories.
- Validation of extraction coverage.
- Export of the processed dataset for Power BI dashboard development and machine learning tasks.

The resulting dataset serves as the foundation for business intelligence reporting and predictive modeling in the next stages of the project.